In [2]:
#Import dependency list

import os
from glob import glob

import nibabel as nib
import numpy as np
import pandas as pd
import scipy
from time import time
import datetime
import json
from itertools import product
from xgboost import XGBClassifier
import seaborn as sns
import re, string
import csv
import matplotlib.pyplot as plt 
from matplotlib.ticker import PercentFormatter
from joblib import Parallel, delayed
from re import search
import sklearn as sk
from sklearn.feature_selection import SelectKBest,f_classif
from sklearn.ensemble import RandomForestClassifier
from neuroCombat import neuroCombat
import sklearn
from sklearn import svm
from pycombat import Combat
from sklearn.metrics.pairwise import pairwise_kernels
from sklearn.model_selection import KFold,train_test_split,StratifiedShuffleSplit, StratifiedKFold, LeaveOneGroupOut, cross_validate, GridSearchCV, cross_val_predict
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, KernelCenterer, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC, LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report,confusion_matrix,auc,roc_curve
from sklearn.utils import shuffle
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
import nilearn as nil
from nilearn import plotting
from nilearn.masking import compute_epi_mask
from nilearn.plotting import plot_roi
from nilearn import datasets
from libsvm.svmutil import *
from xgboost import XGBClassifier
from pprint import pprint
#from sklearn_rvm import EMRVC, EMRVR
from scipy.stats import mode
import os.path as osp
from imblearn.under_sampling import RandomUnderSampler
from numpy import mean,std
from confidenceinterval import roc_auc_score as aucci
from confidenceinterval import precision_score, recall_score, f1_score
from confidenceinterval import accuracy_score,ppv_score,npv_score,tpr_score,fpr_score,tnr_score
from confidenceinterval.bootstrap import bootstrap_ci

In [3]:

# Multisite classification pipeline with LOSO and ComBat-stratified CV. Example run: train_predict(data_X,data_Y,array_of_site_membership,combat='True' or 'False')


#Primary AUC estimator is the average over folds/sites (instead of pooled) to best evaluate clinical utility

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import shap

from collections import defaultdict

from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import (GridSearchCV, StratifiedKFold,
                                     LeaveOneGroupOut, BaseCrossValidator)
from sklearn.metrics import confusion_matrix, precision_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from imblearn.under_sampling import RandomUnderSampler
from pycombat import Combat


AUC_ESTIMATOR = "fold_mean"   # "fold_mean" (default) or "pooled"
COMPUTE_POOLED = True         
FOLD_WEIGHTING = "equal"      #weighting of auc is done per fold/auc

#Use modified version of DeLong method to calculate confidence intervals (also includes t-based CI and random-effects CI to ensure DeLong is not heavily biased)
FOLD_CI_METHOD = "delong"


#impute nans and scale data using standardscaler
def clean_and_scale(X_train, X_test, scaler=StandardScaler()):
    X_train = np.array(X_train, dtype=float, copy=True)
    X_test = np.array(X_test, dtype=float, copy=True)

    
    cm = np.nanmean(X_train, axis=0)
    cm = np.where(np.isnan(cm), 0.0, cm)
    for X in (X_train, X_test):
        inds = np.where(np.isnan(X))
        X[inds] = np.take(cm, inds[1])

    if scaler:
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
    return X_train, X_test

#Function to apply combat on train and test separately
def apply_combat(X_train, X_test, batch_train, batch_test, covars_train, covars_test):
    combat = Combat()
    X_train = combat.fit_transform(Y=X_train, b=batch_train, X=None, C=covars_train)
    X_test = combat.transform(Y=X_test, b=batch_test, X=None, C=covars_test)
    return X_train, X_test


#Model fitting + evaluation functions

def train_model(X_train, y_train, model, param_grid, inner_cv, n_jobs=16,
                model_type=None, settings=None):
    grid = GridSearchCV(model, param_grid, scoring='balanced_accuracy',
                        cv=inner_cv.split(X_train, y_train), refit=True, n_jobs=n_jobs)
    grid.fit(X_train, y_train)
    return grid.best_estimator_

#Model probability prediction for AUC calc
def _predict_scores(model, X):
    
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return model.predict(X) 

#tied midrank computation used by DeLong method
def _compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T
    return T2

#Calculate delong variance within each fold prediction
def delong_auc_var(y_true, y_scores):
    y_true = np.asarray(y_true).ravel()
    y_scores = np.asarray(y_scores, dtype=float).ravel()
    positives = y_scores[y_true == 1]
    negatives = y_scores[y_true == 0]
    m_pos, n_neg = len(positives), len(negatives)
    if m_pos == 0 or n_neg == 0:
        return np.nan, np.nan

    tie_x = _compute_midrank(positives)
    tie_y = _compute_midrank(negatives)
    tie_z = _compute_midrank(np.concatenate([positives, negatives]))

    auc = (tie_z[:m_pos].sum() - m_pos * (m_pos + 1) / 2) / (m_pos * n_neg)
    # structural components over positives
    v_pos = (tie_z[:m_pos] - tie_x) / n_neg              
    # structural components over negatives
    v_neg = 1.0 - (tie_z[m_pos:] - tie_y) / m_pos        

    if m_pos < 2 or n_neg < 2:
        return float(auc), np.nan
    var = np.var(v_pos, ddof=1) / m_pos + np.var(v_neg, ddof=1) / n_neg
    return float(auc), float(var)



#Default: delong AUC CIs, but per site instead of pooled across (for clinical utility)
def delong_fold_mean_ci(fold_aucs, fold_vars, weights=None, confidence_level=0.95):

    aucarray = np.asarray(fold_aucs, dtype=float)
    variances = np.asarray(fold_vars, dtype=float)
    mask = ~(np.isnan(aucarray) | np.isnan(variances))
    aucarray, variances = aucarray[mask], variances[mask]
    if len(aucarray) == 0:
        return np.nan, (np.nan, np.nan), np.nan
    #we use weights=none, but optional implementation for weighing by sample size
    if weights is None:
        w = np.ones(len(aucarray)) / len(aucarray)
    else:
        w = np.asarray(weights, dtype=float)[mask]
        w = w / w.sum()

    #Calculate mean as weights * Auc values per fold 
    mean = float(np.sum(w * aucarray))
    stde = float(np.sqrt(np.sum(w ** 2 * variances)))
    z = stats.norm.ppf(0.5 + confidence_level / 2)
    return mean, (mean - z * stde, mean + z * stde), stde


#Additional sanity check: CI calculated from random effects
def random_effects_fold_ci(fold_aucs, fold_vars, confidence_level=0.95):

    a = np.asarray(fold_aucs, dtype=float)
    v = np.asarray(fold_vars, dtype=float)
    mask = ~(np.isnan(a) | np.isnan(v))
    a, v = a[mask], v[mask]
    k = len(a)
    if k < 2:
        return np.nan, (np.nan, np.nan), np.nan, np.nan

    wf = 1.0 / v
    mu_f = np.sum(wf * a) / np.sum(wf)
    Q = float(np.sum(wf * (a - mu_f) ** 2))
    C = np.sum(wf) - np.sum(wf ** 2) / np.sum(wf)
    tau2 = max(0.0, (Q - (k - 1)) / C) if C > 0 else 0.0

    wr = 1.0 / (v + tau2)
    mean = float(np.sum(wr * a) / np.sum(wr))
    se = float(np.sqrt(1.0 / np.sum(wr)))
    z = stats.norm.ppf(0.5 + confidence_level / 2)
    return mean, (mean - z * se, mean + z * se), se, tau2

#Sanity check: t-based CI
def fold_mean_ci(values, weights=None, confidence_level=0.95):
    vals = np.asarray(values, dtype=float)
    mask = ~np.isnan(vals)
    vals = vals[mask]
    n = len(vals)

    if n == 0:
        return np.nan, (np.nan, np.nan), np.nan, np.nan, 0
    if n == 1:
        return float(vals[0]), (np.nan, np.nan), np.nan, np.nan, 1

    if weights is None:
        mean = float(vals.mean())
        sd = float(vals.std(ddof=1))
        sem = sd / np.sqrt(n)
    else:
        w = np.asarray(weights, dtype=float)[mask]
        w = w / w.sum()
        mean = float(np.sum(w * vals))
        # weighted variance, then effective-n correction (Kish)
        var = float(np.sum(w * (vals - mean) ** 2) / (1 - np.sum(w ** 2)))
        sd = np.sqrt(var)
        n_eff = 1.0 / np.sum(w ** 2)
        sem = sd / np.sqrt(n_eff)
        n = n_eff

    tcrit = stats.t.ppf(0.5 + confidence_level / 2, df=max(n - 1, 1))
    return mean, (mean - tcrit * sem, mean + tcrit * sem), sd, sem, int(round(n))


#def safe_auc(y_true, y_scores):
#    y_true = np.asarray(y_true).ravel()
#    if len(np.unique(y_true)) < 2:
#        return np.nan
#    return float(roc_auc_score(y_true, np.asarray(y_scores).ravel()))

#Model evaluation function. Returns AUC,PPV,NPV,sensitivity and specificity
def evaluate_model(y_test, y_pred_labels, y_scores):
    y_test = np.asarray(y_test).ravel()
    y_pred_labels = np.asarray(y_pred_labels).ravel()
    y_scores = np.asarray(y_scores).ravel()
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_labels, labels=[0, 1]).ravel()
    return {
        "AUC": roc_auc_score(y_test, y_scores),
        "PPV": float(precision_score(y_test, y_pred_labels, zero_division=0)),
        "NPV": float(tn / (tn + fn)) if (tn + fn) else 0.0,
        "Sensitivity": float(tp / (tp + fn)) if (tp + fn) else 0.0,
        "Specificity": float(tn / (tn + fp)) if (tn + fp) else 0.0,
    }


# SHAP functions

def extract_fold_shap(best_model, X_train, X_test, model_type):
    if model_type in ('svm', 'logreg'):
        background = shap.kmeans(X_train, min(10, len(X_train)))
        explainer = shap.KernelExplainer(
            lambda x: _predict_scores(best_model, x), background
        )
        sv = np.asarray(explainer.shap_values(X_test))
    elif model_type in ('rf', 'xgb'):
        explainer = shap.TreeExplainer(best_model)
        sv = explainer.shap_values(X_test, check_additivity=False)
        if isinstance(sv, list):
            sv = np.asarray(sv[1])            # older shap: [class0, class1]
        else:
            sv = np.asarray(sv)
            if sv.ndim == 3:                  # shap >= 0.45: (n, f, n_classes)
                sv = sv[:, :, 1]
    else:
        return None

    if sv.ndim == 3 and sv.shape[-1] == 1:
        sv = sv[:, :, 0]
    if sv.shape != X_test.shape:
        raise ValueError(f"SHAP shape {sv.shape} != X_test shape {X_test.shape}")
    return sv

#Calculate shap means per fold
def fold_shap_means(shap_values, feature_names):
   
    sv = np.asarray(shap_values)
    if sv.ndim != 2:
        raise ValueError(f"Expected 2-D SHAP array, got shape {sv.shape}")
    if sv.shape[1] != len(feature_names):
        raise ValueError(f"SHAP has {sv.shape[1]} features but "
                         f"{len(feature_names)} names were given")
    return pd.DataFrame(
        {"mean_abs_shap": np.abs(sv).mean(axis=0),
         "mean_signed_shap": sv.mean(axis=0)},
        index=pd.Index(feature_names, name="feature"),
    )

#Collect top 25 absolute shap values across folds
def aggregate_shap(shap_folds, model_type, cv_name, combat=False, sampling=False,
                   k_best=None, top_n=25, min_folds=1, plot=True,
                   rank_by="mean_abs_shap_zerofill"):

    if not shap_folds:
        print("No SHAP values collected - nothing to aggregate.")
        return None

    abs_df = pd.concat([f["mean_abs_shap"].rename(i)
                        for i, f in enumerate(shap_folds)], axis=1)
    sign_df = pd.concat([f["mean_signed_shap"].rename(i)
                         for i, f in enumerate(shap_folds)], axis=1)

    n_sel = abs_df.notna().sum(axis=1)
    summary = pd.DataFrame({
        "n_folds_selected": n_sel,
        "n_folds_total": len(shap_folds),
        "mean_abs_shap": abs_df.mean(axis=1),
        "sd_abs_shap": abs_df.std(axis=1, ddof=1),
        "mean_abs_shap_zerofill": abs_df.fillna(0.0).mean(axis=1),
        "mean_signed_shap": sign_df.mean(axis=1),
    })
    summary["sem_abs_shap"] = summary["sd_abs_shap"] / np.sqrt(summary["n_folds_selected"])
    summary = summary.sort_values(rank_by, ascending=False)

    stem = (f"{model_type}_cv-{cv_name}_combat-{combat}"
            f"_sampling-{sampling}_kbest-{k_best}")
    summary.to_csv(f"shap_summary_{stem}.csv")
    abs_df.to_csv(f"shap_perfold_abs_{stem}.csv")

    plot_df = summary[summary["n_folds_selected"] >= min_folds].head(top_n).iloc[::-1]
    if plot and len(plot_df):
        plt.figure(figsize=(8, max(4, 0.3 * len(plot_df))))
        plt.barh(plot_df.index, plot_df[rank_by],
                 xerr=plot_df["sem_abs_shap"].fillna(0.0), capsize=3, color="skyblue")
        plt.xlabel("Mean |SHAP| (averaged across folds, +/- SEM across folds)")
        title = f"{model_type.upper()}, CV: {cv_name}, Combat: {'ON' if combat else 'OFF'}"
        title += f", Sampling: {'ON' if sampling else 'OFF'}"
        if k_best:
            title += f", KBest: {k_best}"
        plt.title(title)
        plt.tight_layout()
        plt.savefig(f"shap_bar_{stem}.png", dpi=300, bbox_inches="tight")
        plt.show()

    return summary


##Permutation test function, 100 permutations by default. Essentially copies original train_predict call but with shuffled labels
def permutation_test(data, variable, sites,  true_auc,
                     model_type, sampling, covars, scaler, k_best, combat,
                     n_permutations=100):
    
    aucs = []
    rng = np.random.RandomState()
    for i in range(n_permutations):
        permuted_variable = variable.copy()
        #ensure labels are permuted within site
        for site in np.unique(sites):
            mask = sites == site
            permuted_variable[mask] = rng.permutation(variable[mask])

        auc = train_predict(data, permuted_variable, sites, 
                            model_type=model_type, combat=combat, sampling=sampling,
                            covars=covars, scaler=scaler, k_best=k_best,
                            return_auc_only=True, shap_calc=False, verbose=False)
        aucs.append(auc)
        if (i + 1) % 10 == 0:
            print(f"  permutation {i + 1}/{n_permutations}, "
                  f"null AUC so far: mean={np.nanmean(aucs):.3f}, "
                  f"sd={np.nanstd(aucs):.3f}")

    aucs = np.asarray(aucs, dtype=float)
    valid = aucs[~np.isnan(aucs)]
    #Calculate p value as sum of permuted AUCs being higher or equal to orginally obtained AUC
    p_value = float((np.sum(valid >= true_auc) + 1) / (len(valid) + 1))
    #Monte carlo standard error
    mc_se = float(np.sqrt(p_value * (1 - p_value) / max(len(valid), 1)))

    print(f"Permutation test: observed AUC = {true_auc:.3f}, "
          f"null mean = {valid.mean():.3f} (sd {valid.std(ddof=1):.3f}), "
          f"n = {len(valid)}")
    print(f"  p = {p_value:.4f}  (Monte Carlo SE ~ {mc_se:.4f}; "
          f"min attainable p = {1/(len(valid)+1):.4f})")
    if len(valid) < 1000:
        low = max(p_value - 1.96 * mc_se, 0.0)
        high = min(p_value + 1.96 * mc_se, 1.0)
        print(f"  NOTE: only {len(valid)} permutations - p is estimated to "
              f"within roughly [{low:.3f}, {high:.3f}]. Adequate for screening; "
              f"increase before reporting a p near the 0.05 boundary.")
    return p_value, valid


#quick Ci calc for vals
def _t_ci(vals):
    vals = np.asarray(vals, dtype=float)
    vals = vals[~np.isnan(vals)]
    n = len(vals)
    if n < 2:
        return (np.nan, np.nan)
    return stats.t.interval(0.95, n - 1, loc=np.mean(vals),
                            scale=np.std(vals, ddof=1) / np.sqrt(n))

#Old function to plot metrics output, no longer necessary
#def plot_metrics_bar(metrics, model_type, cv_name, auc_ci, combat=False,
#                     sampling=False, k_best=None):
#    labels = ["AUC", "Sensitivity", "Specificity", "NPV", "PPV"]
#    means = [np.nanmean(metrics[m]) for m in labels]
#    errors = [0.0] * len(means)
#    if np.all(np.isfinite(auc_ci)):
#        errors[0] = (auc_ci[1] - auc_ci[0]) / 2

 #   plt.figure(figsize=(8, 6))
 #   bars = plt.bar(labels, means, yerr=errors, capsize=5, color='skyblue')
 #   plt.ylim(0, 1)
 #   plt.axhline(0.5, color='grey', linestyle='--', linewidth=1)
 #   title = f"Model: {model_type.upper()}, CV: {cv_name}"
 #   title += f", Combat: {'ON' if combat else 'OFF'}"
 #   title += f", Sampling: {'ON' if sampling else 'OFF'}"
 #   if k_best:
 #       title += f", KBest: {k_best}"
 #   plt.title(title)
 #   plt.ylabel("Score (fold-averaged)")
 #   for bar in bars:
 #       height = bar.get_height()
 #       plt.text(bar.get_x() + bar.get_width() / 2., height + 0.02,
 #                f"{height:.2f}", ha='center')
 #   plt.tight_layout()
 #   plt.show()#
#
 #   avg_metrics = {f"{k}_foldmean": np.nanmean(v) for k, v in metrics.items()}
  #  ci_metrics = {}
   # for m in labels:
   #     lo, hi = _t_ci(metrics[m])
   #     ci_metrics[f"{m}_CI_L"], ci_metrics[f"{m}_CI_H"] = lo, hi

#    fname = (f"average_metrics_{model_type}_cv-{cv_name}_combat-{combat}"
#             f"_sampling-{sampling}_kbest-{k_best}.csv")
#    pd.DataFrame([{**avg_metrics, **ci_metrics}]).to_csv(fname, index=False)



#Custom 5-fold to ensure all sites are equally distributed in outer_cv. Prevents insufficient samples from smallest site (4) from ending up in test set
class StratifiedSiteKFold(BaseCrossValidator):
    def __init__(self, n_splits=5, site_labels=None, rare_site=None, random_state=42):
        self.n_splits = n_splits
        self.site_labels = site_labels
        self.rare_site = rare_site
        self.random_state = random_state

    def split(self, X, y=None, groups=None):
        rng = np.random.RandomState(self.random_state)
        site_labels = np.array(self.site_labels)
        rare_mask = site_labels == self.rare_site

        #random shuffle of rare and common site samples
        rare_indices = rng.permutation(np.where(rare_mask)[0])
        common_indices = rng.permutation(np.where(~rare_mask)[0])

        folds = [[] for _ in range(self.n_splits)]
        for i, idx in enumerate(rare_indices):
            folds[i % self.n_splits].append(idx)
        for i, idx in enumerate(common_indices):
            folds[i % self.n_splits].append(idx)

        for i in range(self.n_splits):
            test_idx = np.array(folds[i])
            train_idx = np.array([idx for j, fold in enumerate(folds)
                                  if j != i for idx in fold])
            yield train_idx, test_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits



#Grid settings, also contains deprecated logreg function
def _get_model_and_grid(model_type):
    if model_type == 'svm':
        return svm.SVC(), {
            'class_weight': ['balanced'],
            'C': [0.1, 1, 10, 100, 1000],
            'gamma': [1, 0.1, 0.01, 0.001, 0.0001],
            'kernel': ['rbf', 'linear'],
        }
    if model_type == 'rf':
        return RandomForestClassifier(), {
            'class_weight': ['balanced'],
            'bootstrap': [True],
            'max_depth': [None],
            'max_features': [10, 50, 100, 300],
            'min_samples_leaf': [1, 5, 10],
            'min_samples_split': [2, 10, 20],
            'n_estimators': [100, 200, 300, 1000],
        }
    #if model_type == 'logreg':
    #    model = LogisticRegression(penalty='l1', solver='liblinear',
     #                              class_weight='balanced', max_iter=500)
    #    return model, {'C': [0.01, 0.1, 1, 10, 100]}
    #raise ValueError("Invalid model_type: choose 'svm', 'rf', or 'logreg'")



#Overarching training and CV function. Default is LOSO (leave one group out with group being site). Combat is activated with combat=True
def train_predict(original_data, variable, sites, 
                  model_type='svm', combat=False, sampling=False,
                  covars=None, scaler=StandardScaler(), k_best=None,
                  return_auc_only=False, permute=False, auto_permute=True,
                  shap_calc=False, n_permutations=None,verbose=True):

    data = np.asarray(original_data)
    variable = np.asarray(variable).ravel()
    sites = np.asarray(sites).ravel()
    if covars is not None:
        covars = np.asarray(covars)

    #ensure X,Y and site list is equal shape!
    if data.shape[0] != len(variable) or data.shape[0] != len(sites):
        raise ValueError(f"Length mismatch: X={data.shape[0]}, "
                         f"y={len(variable)}, sites={len(sites)}")
    #Extract column names for shap
    all_feature_names = np.asarray(
        original_data.columns if hasattr(original_data, "columns")
        else [f"f{i}" for i in range(data.shape[1])]
    )
    #important dependency: combat or LOSO?
    if combat:
        uniq, counts = np.unique(sites, return_counts=True)
        rare = uniq[np.argmin(counts)]
        outer_cv = StratifiedSiteKFold(n_splits=5, site_labels=sites,
                                       rare_site=rare, random_state=42)
        cv_name = "Combat"
    else:
        outer_cv = LeaveOneGroupOut()
        cv_name = "Leave-One-Site-Out"
    #Stratify by outcome in inner fold to hopefully prevent unbalanced learning
    inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=12345)
    
    #Initiate lists
    metrics = defaultdict(list)
    fold_sizes, fold_ids, fold_auc_vars = [], [], []
    shap_folds = []
    totalytest, totalypred, totalyscores = [], [], []

    #Start training
    for fold_i, (train_idx, test_idx) in enumerate(outer_cv.split(data, variable, sites)):
        X_train, X_test = data[train_idx], data[test_idx]
        y_train, y_test = variable[train_idx], variable[test_idx]

        X_train, X_test = clean_and_scale(X_train, X_test, scaler)

        if combat:
            X_train, X_test = apply_combat(
                X_train, X_test, sites[train_idx], sites[test_idx],
                covars[train_idx], covars[test_idx])
            X_train, X_test = clean_and_scale(X_train, X_test, scaler)

        feature_names = all_feature_names
        if k_best:
            selector = SelectKBest(f_classif, k=k_best)
            X_train = selector.fit_transform(X_train, y_train)
            X_test = selector.transform(X_test)
            feature_names = all_feature_names[selector.get_support()]

        #Random undersampling option for majority class (default=off)
        if sampling:
            rus = RandomUnderSampler(random_state=0)
            X_train, y_train = rus.fit_resample(X_train, y_train)

        #Obtain grid search parameters
        model, param_grid = _get_model_and_grid(model_type)
        settings = {"cv_name": cv_name, "combat": combat,
                    "sampling": sampling, "k_best": k_best}

        best_model = train_model(X_train, y_train, model, param_grid, inner_cv,
                                 model_type=model_type, settings=settings)

        #Predict probability outcome from grid-searched model
        y_scores = _predict_scores(best_model, X_test)
        #Predict binary outcome from grid-searched model
        y_pred = best_model.predict(X_test)

        fold_metrics = evaluate_model(y_test, y_pred, y_scores)
        for k, v in fold_metrics.items():
            metrics[k].append(v)

        #Obtain DeLong variance
        _, fold_var = delong_auc_var(y_test, y_scores)
        fold_auc_vars.append(fold_var)

        fold_sizes.append(len(test_idx))
        fold_ids.append(np.unique(sites[test_idx]).tolist())

        #Save predictions to list for possible pooled AUC preds
        totalytest.append(np.asarray(y_test).ravel())
        totalypred.append(np.asarray(y_pred).ravel())
        totalyscores.append(np.asarray(y_scores).ravel())

        if shap_calc and not return_auc_only:
            sv = extract_fold_shap(best_model, X_train, X_test, model_type)
            if sv is not None:
                shap_folds.append(fold_shap_means(sv, feature_names))

    totalytest = np.concatenate(totalytest)
    totalypred = np.concatenate(totalypred)
    totalyscores = np.concatenate(totalyscores)

    #Calculate FOLd-averaged AUC and DeLong CI with fallbacks to two other CI types
    weights = fold_sizes if FOLD_WEIGHTING == "n_test" else None

    fm_auc_t, fm_ci_t, fm_sd, fm_sem_t, n_used = fold_mean_ci(
        metrics["AUC"], weights=weights)
    fm_auc_dl, fm_ci_dl, fm_se_dl = delong_fold_mean_ci(
        metrics["AUC"], fold_auc_vars, weights=weights)
    fm_auc_re, fm_ci_re, fm_se_re, tau2 = random_effects_fold_ci(
        metrics["AUC"], fold_auc_vars)

    if FOLD_CI_METHOD == "delong":
        fm_auc, fm_ci, fm_se = fm_auc_dl, fm_ci_dl, fm_se_dl
    elif FOLD_CI_METHOD == "random_effects":
        fm_auc, fm_ci, fm_se = fm_auc_re, fm_ci_re, fm_se_re
    else:
        fm_auc, fm_ci, fm_se = fm_auc_t, fm_ci_t, fm_sem_t

    #sanity check: also calculate pooled AUC
    pooled_auc, pooled_ci = np.nan, (np.nan, np.nan)
    if COMPUTE_POOLED:
        try:
            pooled_auc, _ci = aucci(totalytest, totalyscores, confidence_level=0.95)
            _ci = np.asarray(_ci, dtype=float).ravel()
            pooled_auc, pooled_ci = float(pooled_auc), (float(_ci[0]), float(_ci[1]))
        except (NameError, Exception):
            pass

    #averaged or pooled?
    final_auc = fm_auc if AUC_ESTIMATOR == "fold_mean" else pooled_auc
    final_ci = fm_ci if AUC_ESTIMATOR == "fold_mean" else pooled_ci

    #Only return auc if permutation test running
    if return_auc_only:
        return final_auc

    if verbose:
        print(f"--- {model_type.upper()} | {cv_name} | combat={combat} | "
              f"sampling={sampling} | k_best={k_best} ---")
        print(f"n = {len(totalytest)} held-out subjects across {len(metrics['AUC'])} "
              f"folds (sizes {fold_sizes})")
        print(f"PRIMARY  fold-mean AUC: {fm_auc:.3f}, 95% CI [{fm_ci[0]:.3f}, "
              f"{fm_ci[1]:.3f}]  (method={FOLD_CI_METHOD}, SE {fm_se:.3f}, "
              f"weighting={FOLD_WEIGHTING})")
        print(f"         per-fold AUCs: "
              f"{['%.3f' % a for a in metrics['AUC']]}")
        print(f"         per-fold DeLong SE: "
              f"{['%.3f' % np.sqrt(v) if np.isfinite(v) else 'nan' for v in fold_auc_vars]}")
        print(f"         alt CIs -- DeLong [{fm_ci_dl[0]:.3f}, {fm_ci_dl[1]:.3f}] | "
              f"t across folds [{fm_ci_t[0]:.3f}, {fm_ci_t[1]:.3f}] | "
              f"random-effects [{fm_ci_re[0]:.3f}, {fm_ci_re[1]:.3f}] (tau^2={tau2:.4f})")
        if np.isfinite(pooled_auc):
            print(f"SECOND'Y pooled DeLong AUC: {pooled_auc:.3f}, 95% CI "
                  f"[{pooled_ci[0]:.3f}, {pooled_ci[1]:.3f}]")
        for m in ["PPV", "NPV", "Sensitivity", "Specificity"]:
            vals = np.asarray(metrics[m], dtype=float)
            lo, hi = _t_ci(vals)
            print(f"{m:12s} {np.nanmean(vals):.2f} (+-{np.nanstd(vals, ddof=1):.2f})"
                  f"  95% CI [{lo:.2f}, {hi:.2f}]")

    shap_summary = None
    if shap_calc:
        shap_summary = aggregate_shap(shap_folds, model_type, cv_name,
                                      combat=combat, sampling=sampling,
                                      k_best=k_best)

    p_value, null_aucs = None, None
    #permutation set to run if lower bound of CI is above 0.5, even runs when permute is accidentally set to False
    auto_fired = bool(auto_permute and np.isfinite(final_ci[0]) and final_ci[0] > 0.5)
    if permute or auto_fired:
        n_perm = n_permutations or 100
        if auto_fired and not permute:
            print(f"Lower 95% CI bound ({final_ci[0]:.3f}) is above 0.5 "
                  f"-> running permutation test automatically.")
        print(f"Starting permutation test ({n_perm} permutations, "
              f"statistic = {AUC_ESTIMATOR})...")
        p_value, null_aucs = permutation_test(
            data, variable, sites,  final_auc, model_type, sampling,
            covars, scaler, k_best, combat, n_permutations=n_perm)

    return {
        "auc": final_auc,
        "ci": final_ci,
        "estimator": AUC_ESTIMATOR,
        "fold_mean_auc": fm_auc,
        "fold_mean_ci": fm_ci,
        "fold_ci_method": FOLD_CI_METHOD,
        "fold_mean_ci_delong": fm_ci_dl,
        "fold_mean_ci_t": fm_ci_t,
        "fold_mean_ci_random_effects": fm_ci_re,
        "tau2": tau2,
        "fold_sd": fm_sd,
        "fold_se": fm_se,
        "per_fold_auc": list(metrics["AUC"]),
        "per_fold_auc_var": fold_auc_vars,
        "fold_sizes": fold_sizes,
        "fold_sites": fold_ids,
        "pooled_auc": pooled_auc,
        "pooled_ci": pooled_ci,
        "metrics": dict(metrics),
        "shap_summary": shap_summary,
        "p_value": p_value,
        "null_aucs": null_aucs,
    }